# Training Data Subsampling

This notebook provides a simple way to subsample training data from JSON files.

- If a `data_source` field exists, it performs **stratified sampling** (maintains proportions across sources)
- Otherwise, it performs **random sampling**

## Supported datasets:
- `longvideo-reason.json` (7,019 examples)
- `vidchapters.json` (10,571 examples)


In [2]:
import json
import random
from pathlib import Path
from collections import Counter

# Set random seed for reproducibility
random.seed(42)


In [3]:
def subsample_data(data, sample_ratio=0.1, stratify_field='data_source'):
    """
    Subsample data with optional stratification.
    
    Args:
        data: List of examples (dictionaries)
        sample_ratio: Proportion of data to sample (0.0 to 1.0)
        stratify_field: Field to use for stratified sampling (if exists)
    
    Returns:
        Subsampled list of examples
    """
    total_examples = len(data)
    target_count = int(total_examples * sample_ratio)
    
    print(f"Total examples: {total_examples}")
    print(f"Target sample size: {target_count} ({sample_ratio*100:.1f}%)")
    
    # Check if stratify_field exists in the data
    has_stratify_field = stratify_field in data[0] if data else False
    
    if has_stratify_field:
        # Stratified sampling
        print(f"\nPerforming stratified sampling by '{stratify_field}'")
        
        # Group by stratify_field
        groups = {}
        for example in data:
            source = example.get(stratify_field, 'unknown')
            if source not in groups:
                groups[source] = []
            groups[source].append(example)
        
        # Show distribution
        print("\nOriginal distribution:")
        for source, examples in sorted(groups.items()):
            print(f"  {source}: {len(examples)} ({len(examples)/total_examples*100:.1f}%)")
        
        # Sample from each group proportionally
        sampled = []
        for source, examples in groups.items():
            group_ratio = len(examples) / total_examples
            group_target = int(target_count * group_ratio)
            group_sample = random.sample(examples, min(group_target, len(examples)))
            sampled.extend(group_sample)
        
        # If we're short, randomly sample more to reach target
        if len(sampled) < target_count:
            remaining = target_count - len(sampled)
            all_ids = {ex['id'] for ex in sampled}
            remaining_examples = [ex for ex in data if ex['id'] not in all_ids]
            additional = random.sample(remaining_examples, min(remaining, len(remaining_examples)))
            sampled.extend(additional)
        
        print(f"\nSampled distribution:")
        sampled_groups = Counter(ex.get(stratify_field, 'unknown') for ex in sampled)
        for source, count in sorted(sampled_groups.items()):
            print(f"  {source}: {count} ({count/len(sampled)*100:.1f}%)")
    else:
        # Random sampling
        print(f"\nPerforming random sampling (no '{stratify_field}' field found)")
        sampled = random.sample(data, min(target_count, len(data)))
    
    print(f"\nFinal sample size: {len(sampled)}")
    return sampled


## Configuration

Set your desired sample ratio (0.0 to 1.0)


In [11]:
# Configuration
SAMPLE_RATIO = 0.3  # Sample 10% of the data (change as needed)

# Paths
# DATA_DIR = Path("data/MultiTaskVideoReasoning/MTVR_Tool_CoT")
DATA_DIR = Path("data/MultiTaskVideoReasoning/MTVR_Tool_RL")


## 1. Subsample longvideo-reason.json


In [12]:
# Load longvideo-reason.json
input_file = DATA_DIR / "longvideo-reason.json"
print(f"Loading {input_file}...")

with open(input_file, 'r') as f:
    longvideo_data = json.load(f)

print(f"✓ Loaded {len(longvideo_data)} examples")


Loading data/MultiTaskVideoReasoning/MTVR_Tool_RL/longvideo-reason.json...
✓ Loaded 8000 examples


In [13]:
# Subsample the data
print(f"\n{'='*60}")
print("LONGVIDEO-REASON SUBSAMPLING")
print(f"{'='*60}")
longvideo_sampled = subsample_data(longvideo_data, sample_ratio=SAMPLE_RATIO)

# Save subsampled data
output_file = DATA_DIR / f"longvideo-reason_sampled_{int(SAMPLE_RATIO*100)}pct.json"
with open(output_file, 'w') as f:
    json.dump(longvideo_sampled, f, indent=2)

print(f"\n✓ Saved to: {output_file}")



LONGVIDEO-REASON SUBSAMPLING
Total examples: 8000
Target sample size: 2400 (30.0%)

Performing stratified sampling by 'data_source'

Original distribution:
  video_r1/multiple_choice: 8000 (100.0%)

Sampled distribution:
  video_r1/multiple_choice: 2400 (100.0%)

Final sample size: 2400

✓ Saved to: data/MultiTaskVideoReasoning/MTVR_Tool_RL/longvideo-reason_sampled_30pct.json


## 2. Subsample vidchapters.json


In [14]:
# Load vidchapters.json
input_file = DATA_DIR / "vidchapters.json"
print(f"Loading {input_file}...")

with open(input_file, 'r') as f:
    vidchapters_data = json.load(f)

print(f"✓ Loaded {len(vidchapters_data)} examples")


Loading data/MultiTaskVideoReasoning/MTVR_Tool_RL/vidchapters.json...
✓ Loaded 7413 examples


In [15]:
# Subsample the data
print(f"\n{'='*60}")
print("VIDCHAPTERS SUBSAMPLING")
print(f"{'='*60}")
vidchapters_sampled = subsample_data(vidchapters_data, sample_ratio=SAMPLE_RATIO)

# Save subsampled data
output_file = DATA_DIR / f"vidchapters_sampled_{int(SAMPLE_RATIO*100)}pct.json"
with open(output_file, 'w') as f:
    json.dump(vidchapters_sampled, f, indent=2)

print(f"\n✓ Saved to: {output_file}")



VIDCHAPTERS SUBSAMPLING
Total examples: 7413
Target sample size: 2223 (30.0%)

Performing stratified sampling by 'data_source'

Original distribution:
  vidchapter: 7413 (100.0%)

Sampled distribution:
  vidchapter: 2223 (100.0%)

Final sample size: 2223

✓ Saved to: data/MultiTaskVideoReasoning/MTVR_Tool_RL/vidchapters_sampled_30pct.json


## Summary


In [16]:
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Sample ratio: {SAMPLE_RATIO*100:.1f}%")
print(f"\nOutput files:")
print(f"  1. longvideo-reason_sampled_{int(SAMPLE_RATIO*100)}pct.json")
print(f"     Original: {len(longvideo_data)} → Sampled: {len(longvideo_sampled)}")
print(f"  2. vidchapters_sampled_{int(SAMPLE_RATIO*100)}pct.json")
print(f"     Original: {len(vidchapters_data)} → Sampled: {len(vidchapters_sampled)}")
print(f"\n✓ All files saved in: {DATA_DIR}")
print(f"{'='*60}")



SUMMARY
Sample ratio: 30.0%

Output files:
  1. longvideo-reason_sampled_30pct.json
     Original: 8000 → Sampled: 2400
  2. vidchapters_sampled_30pct.json
     Original: 7413 → Sampled: 2223

✓ All files saved in: data/MultiTaskVideoReasoning/MTVR_Tool_RL


In [1]:
import random
import os

random.seed(2025)  # fix seed for reproducibility

data_dir = "/data/user_data/jamesdin/data/actnet"
in_file = os.path.join(data_dir, "actnet_video_ids.txt")
out_file = os.path.join(data_dir, "actnet_video_ids_500.txt")
k = 500

with open(in_file) as f:
    ids = [line.strip() for line in f if line.strip()]

sample = ids if len(ids) <= k else random.sample(ids, k)

with open(out_file, "w") as f:
    for vid in sample:
        f.write(vid + "\n")